In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ਕਦਮ 1: ਸੰਰਚਨਾਤਮਕ ਆਉਟਪੁੱਟ ਲਈ Pydantic ਮਾਡਲ ਪਰਿਭਾਸ਼ਿਤ ਕਰੋ

ਇਹ ਮਾਡਲ ਉਹ **ਸਕੀਮਾ** ਪੇਸ਼ ਕਰਦੇ ਹਨ ਜੋ ਏਜੰਟ ਵਾਪਸ ਕਰਨਗੇ। Pydantic ਨਾਲ `response_format` ਦੀ ਵਰਤੋਂ ਨਾਲ ਇਹ ਨਿਸ਼ਚਿਤ ਹੁੰਦਾ ਹੈ:
- ✅ ਕਿਸਮ-ਸੁਰੱਖਿਅਤ ਡਾਟਾ ਨਿਕਾਸ
- ✅ ਆਪਣੇ ਆਪ ਮਾਨਤਾ
- ✅ ਮੁਫ਼ਤ-ਪਾਠ ਉੱਤਰਾਂ ਵਿੱਚੋਂ ਕੋਈ ਪਾਰਸਿੰਗ ਗਲਤੀ ਨਹੀਂ
- ✅ ਖੇਤਰਾਂ ਦੇ ਆਧਾਰ 'ਤੇ ਆਸਾਨ ਸ਼ਰਤੀ ਰੂਟਿੰਗ


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ਕਦਮ 2: ਹੋਟਲ ਬੁਕਿੰਗ ਟੂਲ ਬਣਾਓ

ਇਹ ਟੂਲ ਉਹ ਹੈ ਜੋ **availability_agent** ਕਮਰੇ ਉਪਲਬਧ ਹਨ ਜਾਂ ਨਹੀਂ ਜਾਂਚਣ ਲਈ ਕਾਲ ਕਰੇਗਾ। ਅਸੀਂ `@ai_function` ਡੈਕੋਰੇਟਰ ਦੀ ਵਰਤੋਂ ਕਰਦੇ ਹਾਂ ਤਾਂ ਕਿ:
- ਇੱਕ ਪਾਇਥਨ ਫੰਕਸ਼ਨ ਨੂੰ ਏਆਈ-ਕਾਲ ਕਰਨ ਯੋਗ ਟੂਲ ਵਿੱਚ ਬਦਲਿਆ ਜਾ ਸਕੇ
- LLM ਲਈ ਆਟੋਮੈਟਿਕ ਤੌਰ 'ਤੇ JSON ਸਕੀਮਾ ਜਨਰੇਟ ਕੀਤਾ ਜਾ ਸਕੇ
- ਪੈਰਾਮੀਟਰ ਵੈਰੀਫਿਕੇਸ਼ਨ ਨੂੰ ਹੈਂਡਲ ਕੀਤਾ ਜਾ ਸਕੇ
- ਏਜੰਟਾਂ ਦੁਆਰਾ ਆਟੋਮੈਟਿਕ ਕਾਲਿੰਗ ਨੂੰ ਯੋਗ ਬਣਾਇਆ ਜਾ ਸਕੇ

ਇਸ ਡੈਮੋ ਲਈ:
- **ਸਟਾਕਹੋਮ, ਸੀਐਟਲ, ਟੋਕਯੋ, ਲੰਡਨ, ਐਮਸਟਰਡੈਮ** → ਕਮਰੇ ਉਪਲਬਧ ✅
- **ਹੋਰ ਸਾਰੇ ਸ਼ਹਿਰ** → ਕੋਈ ਕਮਰੇ ਨਹੀਂ ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ਕਦਮ 3: ਰੂਟਿੰਗ ਲਈ ਸਥਿਤੀ ਫੰਕਸ਼ਨਾਂ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰੋ

ਇਹ ਫੰਕਸ਼ਨ ਏਜੰਟ ਦੇ ਪ੍ਰਤੀਕਿਰਿਆ ਦੀ ਜਾਂਚ ਕਰਦੇ ਹਨ ਅਤੇ ਫ਼ਲੋਵਰਕ ਵਿੱਚ ਕਿਹੜਾ ਰਾਸ਼ਤਾ ਲੈਣਾ ਹੈ ਇਹ ਨਿਰਧਾਰਤ ਕਰਦੇ ਹਨ।

**ਮੁੱਖ ਢਾਂਚਾ:**
1. ਚੈੱਕ ਕਰੋ ਕਿ ਸੰਦਰਸ਼ਨ `AgentExecutorResponse` ਹੈ
2. ਸਰਚਿਤ ਨਿਕਾਸ (Pydantic ਮਾਡਲ) ਨੂੰ ਪਾਰਸ ਕਰੋ
3. ਰੂਟਿੰਗ ਨੂੰ ਨਿਯੰਤਰਤ ਕਰਨ ਲਈ `True` ਜਾਂ `False` ਵਾਪਸ ਕਰੋ

ਵਰਕਫਲੋ ਇਹ ਸਥਿਤੀਆਂ **ਧਾਰਿਆਂ** 'ਤੇ ਮੁਲਾਂਕਣ ਕਰੇਗਾ ਤਾਂ ਜੋ ਅਗਲੇ ਸਿਰਗਰਮ ਕਰਨ ਵਾਲੇ ਨੂੰ ਕੌਣ ਕਾਲ ਕਰਨਾ ਹੈ ਇਹ ਫੈਸਲਾ ਕੀਤਾ ਜਾ ਸਕੇ।


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## ਕਦਮ 4: ਕਸਟਮ ਡਿਸਪਲੇ ਇਕਜ਼ੀਕيوਟਰ ਬਣਾਓ

ਇਕਜ਼ੀਕيوਟਰ ਵਹਾਵਾ ਕੰਪੋਨੈਂਟ ਹੁੰਦੇ ਹਨ ਜੋ ਰੂਪਾਂਤਰਣ ਜਾਂ ਸਾਈਡ ਪ੍ਰਭਾਵ ਕਰਦੇ ਹਨ। ਅਸੀਂ ਆਖਰੀ ਨਤੀਜੇ ਨੂੰ ਦਰਸਾਉਣ ਲਈ `@executor` ਡੈਕੋਰੇਟਰ ਦੀ ਵਰਤੋਂ ਕਰਦੇ ਹਾਂ ਇੱਕ ਕਸਟਮ ਇਕਜ਼ੀਕيوਟਰ ਬਣਾਉਣ ਲਈ।

**ਮੁੱਖ ਸੰਕਲਪ:**
- `@executor(id="...")` - ਕਿਸੇ ਫੰਕਸ਼ਨ ਨੂੰ ਵਹਾਵਾ ਇਕਜ਼ੀਕيوਟਰ ਵਜੋਂ رجسਟਰ ਕਰਦਾ ਹੈ
- `WorkflowContext[Never, str]` - ਇਨਪੁੱਟ/ਆਊਟਪੁੱਟ ਲਈ ਕਿਸਮ ਸੂਚਕ
- `ctx.yield_output(...)` - ਆਖਰੀ ਵਹਾਵਾ ਨਤੀਜਾ ਨੂੰ ਯੀਲਡ ਕਰਦਾ ਹੈ


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## ਕਦਮ 5: ਵਾਤਾਵਰਨ ਵੈਰੀਏਬਲ ਲੋਡ ਕਰੋ

LLM ਕਲਾਇੰਟ ਨੂੰ ਸੰਰਚਿਤ ਕਰੋ। ਇਹ ਉਦਾਹਰਨ ਨੀਚੇ ਨਾਲ ਕੰਮ ਕਰਦੀ ਹੈ:
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — ਪਸੰਦੀਦਾ)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ਚਰਣ 6: ਸੰਰਚਿਤ ਆਉਟਪੁੱਟ ਵਾਲੇ AI ਏਜੰਟ ਬਣਾਓ

ਅਸੀਂ **ਤਿੰਨ ਵਿਸ਼ੇਸ਼ ਏਜੰਟ** ਬਣਾਉਂਦੇ ਹਾਂ, ਹਰ ਇੱਕ ਨੂੰ ਇੱਕ `AgentExecutor` ਵਿੱਚ ਲਪੇਟਿਆ ਗਿਆ ਹੈ:

1. **availability_agent** - ਸੰਦ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਹੋਟਲ ਉਪਲੱਬਧਤਾ ਜਾਂਚਦਾ ਹੈ
2. **alternative_agent** - ਵਿਕਲਪਿਕ ਸ਼ਹਿਰਾਂ ਦਾ ਸੁਝਾਅ ਦਿੰਦਾ ਹੈ (ਜਦੋਂ ਕੋਈ ਕਮਰੇ ਨਹੀਂ)
3. **booking_agent** - ਬੁਕਿੰਗ ਲਈ ਉਤਸ਼ਾਹਤ ਕਰਦਾ ਹੈ (ਜਦੋਂ ਕਮਰੇ ਉਪਲੱਬਧ ਹਨ)

**ਮੁੱਖ ਵਿਸ਼ੇਸ਼ਤਾਵਾਂ:**
- `tools=[hotel_booking]` - ਏਜੰਟ ਨੂੰ ਸੰਦ ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ
- `response_format=PydanticModel` - ਸੰਰਚਿਤ JSON ਆਉਟਪੁੱਟ ਦੀ ਮਜ਼ਬੂਰੀ ਕਰਦਾ ਹੈ
- `AgentExecutor(..., id="...")` - ਵਰਕਫਲੋ ਲਈ ਏਜੰਟ ਨੂੰ ਲਪੇਟਦਾ ਹੈ


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ਕਦਮ 7: ਸਸ਼ਰਤੀ ਕਿਨਾਰਿਆਂ ਨਾਲ ਵਰਕਫਲੋ ਬਣਾਓ

ਹੁਣ ਅਸੀਂ `WorkflowBuilder` ਦੀ ਵਰਤੋਂ ਕਰਦੇ ਹਾਂ ਤਾਂ ਜੋ ਸਸ਼ਰਤੀ ਰੂਟੀੰਗ ਨਾਲ ਗ੍ਰਾਫ ਬਣਾਇਆ ਜਾ ਸਕੇ:

**ਵਰਕਫਲੋ ਸਾਂਰਚਨਾ:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**ਮੁੱਖ ਤਰੀਕੇ:**
- `.set_start_executor(...)` - ਐਂਟਰੀ ਪੌਇੰਟ ਸੈੱਟ ਕਰਦਾ ਹੈ
- `.add_edge(from, to, condition=...)` - ਸਸ਼ਰਤੀ ਕਿਨਾਰੀ ਸ਼ਾਮਲ ਕਰਦਾ ਹੈ
- `.build()` - ਵਰਕਫਲੋ ਨੂੰ ਅੰਤਿਮ ਰੂਪ ਦਿੰਦਾ ਹੈ


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ਧਾਪ 8: ਟੈਸਟ ਕੇਸ 1 ਚਲਾਓ - ਸ਼ਹਿਰ ਬਿਨਾਂ ਉਪਲਬਧਤਾ ਦੇ (ਪੈਰਿਸ)

ਆਓ **ਕੋਈ ਉਪਲਬਧਤਾ ਨਹੀਂ** ਪਾਥ ਨੂੰ ਪਰਖੀਏ, ਪੈਰਿਸ ਵਿੱਚ ਹੋਟਲਾਂ ਦੀ ਬੇਨਤੀ ਕਰਕੇ (ਜਿੱਥੇ ਸਾਡੇ ਅਨੁਕਰਨ ਵਿੱਚ ਕੋਈ ਕਮਰੇ ਨਹੀਂ ਹਨ)।


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## ਕਦਮ 9: ਪਰੀਖਾ ਮਾਮਲਾ 2 ਚਲਾਓ - ਸ਼ਹਿਰ ਸਾਥੀ ਉਪਲਬਧਤਾ ਦੇ (ਸਟਰਕਹੋਮ)

ਹੁਣ ਆਓ **ਉਪਲਬਧਤਾ** ਪਠ ਨੂੰ ਪਰੀਖਾ ਕਰੀਏ ਜਦੋਂ ਅਸੀਂ ਸਟਰਕਹੋਮ ਵਿੱਚ ਹੋਟਲਾਂ ਦੀ ਮੰਗ ਕਰੀਏ (ਜਿਹੜੇ ਸਾਡੇ ਸਿੱਥਲੀਕਰਨ ਵਿੱਚ ਕਮਰੇ ਰੱਖਦੇ ਹਨ).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ਮੁੱਖ ਨਤੀਜੇ ਅਤੇ ਅੱਗੇ ਦੇ ਕਦਮ

### ✅ ਤੁਸੀਂ ਕੀ ਸਿੱਖਿਆ ਹੈ:

1. **WorkflowBuilder ਪੈਟਰਨ**
   - ਐਂਟਰੀ ਪੁਆਇੰਟ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਨ ਲਈ `.set_start_executor()` ਵਰਤੋਂ
   - ਸ਼ਰਤੀ ਰੂਟਿੰਗ ਲਈ `.add_edge(from, to, condition=...)` ਵਰਤੋਂ
   - ਵਰਕਫਲੋ ਨੂੰ ਫਾਈਨਲ ਕਰਨ ਲਈ `.build()` ਕਾਲ ਕਰੋ

2. **ਸ਼ਰਤੀ ਰੂਟਿੰਗ**
   - ਸ਼ਰਤ ਫੰਕਸ਼ਨ `AgentExecutorResponse` ਦੀ ਜਾਂਚ ਕਰਦੇ ਹਨ
   - ਰੂਟਿੰਗ ਫੈਸਲੇ ਕਰਨ ਲਈ ਸਰਚਿੱਟਿਤ ਆਊਟਪੁੱਟ ਨੂੰ ਪਾਰਸ ਕਰੋ
   - ਇੱਕ ਐਜ ਨੂੰ ਸਰਗਰਮ ਕਰਨ ਲਈ `True` ਵਾਪਸ ਕਰੋ, ਛੱਡਣ ਲਈ `False`

3. **ਟੂਲ ਇੰਟਗ੍ਰੇਸ਼ਨ**
   - ਪਾਇਥਨ ਫੰਕਸ਼ਨਾਂ ਨੂੰ ਏਆਈ ਟੂਲਾਂ ਵਿੱਚ ਬਦਲਣ ਲਈ `@ai_function` ਵਰਤੋਂ
   - ਏਜੰਟ ਜਰੂਰਤ ਪੈਣ 'ਤੇ ਟੂਲ ਕਾਲ ਕਰਦੇ ਹਨ
   - ਟੂਲ JSON ਵਾਪਸ ਕਰਦੇ ਹਨ ਜੋ ਏਜੰਟ ਪਾਰਸ ਕਰ ਸਕਦੇ ਹਨ

4. **ਸੰਰਚਿਤ ਆਊਟਪੁੱਟ**
   - ਕਿਸਮ-ਸੁਰੱਖਿਅਤ ਡਾਟਾ ਨਿਕਾਸ ਲਈ ਪਾਇਡੈਂਟਿਕ ਮਾਡਲਾਂ ਵਰਤੋਂ
   - ਏਜੰਟ ਬਣਾਉਂਦੇ ਸਮੇਂ `response_format=MyModel` ਸੈਟ ਕਰੋ
   - ਜਵਾਬਾਂ ਨੂੰ `Model.model_validate_json()` ਨਾਲ ਪਾਰਸ ਕਰੋ

5. **ਕਸਟਮ ਐਗਜ਼ੈਕਿਊਟਰ**
   - ਵਰਕਫਲੋ ਕੰਪੋਨੇਟ ਬਣਾਉਣ ਲਈ `@executor(id="...")` ਵਰਤੋਂ
   - ਐਗਜ਼ੈਕਿਊਟਰ ਡਾਟਾ ਤਬਦੀਲ ਕਰ ਸਕਦੇ ਹਨ ਜਾਂ ਸਾਈਡ ਪ੍ਰਭਾਵ ਕਰ ਸਕਦੇ ਹਨ
   - ਵਰਕਫਲੋ ਨਤੀਜੇ ਪੈਦਾ ਕਰਨ ਲਈ `ctx.yield_output()` ਵਰਤੋਂ

### 🚀 ਅਸਲੀ ਦੁਨੀਆ ਦੇ ਅਰਜ਼ੀਆਂ:

- **ਯਾਤਰਾ ਬੁਕਿੰਗ**: ਉਪਲਬਧਤਾ ਜਾਂਚੋ, ਵਿਕਲਪ ਸੁਝਾਓ, ਵਿਕਲਪਾਂ ਦੀ ਤੁਲਨਾ ਕਰੋ
- **ਗਾਹਕ ਸੇਵਾ**: ਮਸਲੇ ਦੇ ਕਿਸਮ, ਭਾਵਨਾ, ਤਰਜੀਹ ਅਨੁਸਾਰ ਰੂਟ ਕਰੋ
- **ਈ-ਕਾਮਰਸ**: ਇੰਵੇਟਰੀ ਜਾਂਚੋ, ਵਿਕਲਪ ਸੁਝਾਓ, ਆਰਡਰ ਪ੍ਰਕਿਰਿਆ ਕਰੋ
- **ਸਮੱਗਰੀ ਨਿਗਰਾਨੀ**: ਟੌਕਸਿਕਤਾ ਸਕੋਰ, ਉਪਭੋਗਤਾ ਫਲੈਗ ਅਨੁਸਾਰ ਰੂਟ ਕਰੋ
- **ਮਨਜ਼ੂਰੀ ਵਰਕਫਲੋਜ਼**: ਰਕਮ, ਉਪਭੋਗਤਾ ਭੂਮਿਕਾ, ਖ਼ਤਰਾ ਪੱਧਰ ਅਨੁਸਾਰ ਰੂਟ ਕਰੋ
- **ਬਹੁ-ਪਰਤ ਪ੍ਰਕਿਰਿਆ**: ਡਾਟਾ ਗੁਣਵੱਤਾ, ਪੂਰਨਤਾ ਅਨੁਸਾਰ ਰੂਟ ਕਰੋ

### 📚 ਅੱਗੇ ਕਦਮ:

- ਹੋਰ ਜਟਿਲ ਸ਼ਰਤਾਂ ਜੋੜੋ (ਕਈ ਮਾਪਦੰਡ)
- ਵਰਕਫਲੋ ਸਟੇਟ ਪ੍ਰਬੰਧਨ ਨਾਲ ਲੂਪ ਲਾਗੂ ਕਰੋ
- ਦੁਬਾਰਾ ਵਰਤੇ ਜਾ ਸਕਣ ਵਾਲੇ ਕੰਪੋਨੇਟਾਂ ਲਈ ਸਬ-ਵਰਕਫਲੋਜ਼ ਜੋੜੋ
- ਅਸਲੀ APIs ਨਾਲ ਇੰਟਗ੍ਰੇਟ ਕਰੋ (ਹੋਟਲ ਬੁਕਿੰਗ, ਇੰਵੇਟਰੀ ਸਿਸਟਮ)
- ਗਲਤੀ ਸੰਭਾਲ ਅਤੇ ਬੈਕਅਪ ਮਾਰਗ ਜੋੜੋ
- ਅੰਦਰੂਨੀ ਵਿਜ਼ੂਅਲਾਈਜ਼ੇਸ਼ਨ ਟੂਲਾਂ ਨਾਲ ਵਰਕਫਲੋਜ਼ ਦਿਖਾਓ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
